In [1]:
import open_clip

model, preprocess = open_clip.create_model_from_pretrained('hf-hub:laion/CLIP-ViT-g-14-laion2B-s12B-b42K')
tokenizer = open_clip.get_tokenizer('hf-hub:laion/CLIP-ViT-g-14-laion2B-s12B-b42K')

In [2]:
import torch
from PIL import Image
from qdrant_client import QdrantClient, models
import uuid

E0000 00:00:1771165834.087827  965880 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1771165834.087955  965880 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_rejected' registered more than once. Ignoring later registration.
E0000 00:00:1771165834.087958  965880 instrument.cc:563] Metric with name 'grpc.resource_quota.connections_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1771165834.087962  965880 instrument.cc:563] Metric with name 'grpc.resource_quota.instantaneous_memory_pressure' registered more than once. Ignoring later registration.
E0000 00:00:1771165834.087963  965880 instrument.cc:563] Metric with name 'grpc.resource_quota.memory_pressure_control_value' registered more than once. Ignoring later registration.


In [3]:
def search_by_text(query_text, client: QdrantClient, top_k=5):
    """Поиск изображений по текстовому описанию"""
    # 1. Создаем эмбеддинг для текста
    text_tokens = tokenizer([query_text])
    
    with torch.no_grad(), torch.amp.autocast("mps"):
        text_features = model.encode_text(text_tokens)
        text_features /= text_features.norm(dim=-1, keepdim=True)

    # 2. Конвертируем
    text_embedding = text_features.float().numpy()[0].tolist()
    
    # 3. Поиск
    search_results = client.query_points(
        collection_name=collection_name,
        query=text_embedding,
        limit=top_k
    )
    
    return search_results

In [17]:
def insert_image(img, client, collection_name):
    processed_img = preprocess(img).unsqueeze(0)
    with torch.no_grad(), torch.amp.autocast("mps"):
        image_features = model.encode_image(processed_img)
        image_features /= image_features.norm(dim=-1, keepdim=True)

    image_embedding = image_features.cpu().numpy()[0].tolist()
    
    point_id = str(uuid.uuid4())

    vector_size = image_features.shape[-1]

    client.upsert(
        collection_name=collection_name,
        points=[
            models.PointStruct(
                id=point_id,
                vector=image_embedding,
                payload={
                    "image_path": img.filename,
                    "model": "CLIP",
                    "dimensions": vector_size
                }
            )
        ]
    )

In [5]:
# path = "patrick.jpg"
# image = Image.open(path)
# image = preprocess(image).unsqueeze(0)
# client = QdrantClient(":memory:")

In [6]:
# with torch.no_grad(), torch.amp.autocast("mps"):
#     image_features = model.encode_image(image)
#     image_features /= image_features.norm(dim=-1, keepdim=True)

# image_features.dtype

In [8]:
client = QdrantClient(":memory:")
collection_name = "image_vectors"

client.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=1024,
        distance=models.Distance.COSINE
    )
)

True

In [9]:
# image_embedding = image_features.cpu().numpy()[0].tolist()

In [18]:
import os

directory = "./data/"
ar = []
for f in os.listdir(directory):
    if f.endswith(".jpg"):
        filepath = os.path.join(directory, f)
        image = Image.open(filepath)
        insert_image(image, client, collection_name)

In [24]:
results = search_by_text("lenin with heart", client, top_k=3)
[(r['payload'], r['score']) for r in results.model_dump()['points']]

[({'image_path': './data/lenin.jpg', 'model': 'CLIP', 'dimensions': 1024},
  0.3818736483429166),
 ({'image_path': './data/steve.jpg', 'model': 'CLIP', 'dimensions': 1024},
  0.20294584742844216),
 ({'image_path': './data/kislinka.jpg', 'model': 'CLIP', 'dimensions': 1024},
  0.15292547506132498)]